In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from grpo.utils import get_device, seed_everything

seed_everything(0)
get_device(), get_device("cpu")

(device(type='mps'), device(type='cpu'))

In [7]:
from grpo.data import parse_ground_truth, load_gsm8k

parse_ground_truth("blah\n#### 72")        # '72'
parse_ground_truth("blah\n#### 1,200")     # '1200'

train = load_gsm8k("train")
test = load_gsm8k("test")
train[0]["answer"]                         # '72'

'72'

In [8]:
import re
bad = [ex["answer"] for ex in train + test if not re.fullmatch(r"-?\d+(\.\d+)?", ex["answer"])]
len(bad), bad[:10]


(0, [])

In [9]:
from transformers import AutoTokenizer
from grpo.data import build_prompt

for name in ["HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(build_prompt(tok, train[0]["question"]))
    print("=" * 60)


<|im_start|>system
Solve the math problem step by step. On the last line, write the final answer as '#### <number>'.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant

<|im_start|>system
Solve the math problem step by step. On the last line, write the final answer as '#### <number>'.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant



In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from grpo.data import build_prompt
from grpo.utils import get_device

name = "HuggingFaceTB/SmolLM2-135M-Instruct"
device = get_device()
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).to(device)
model.eval()

prompt = build_prompt(tok, train[0]["question"])
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(device)

out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
completion = out[0,  inputs["input_ids"].shape[1]:]
print(tok.decode(completion))

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 14319.52it/s]


Natalia sold 48 clips in April, and she sold half as many clips in May.

#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 4


In [17]:
G = 4
out = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=1.0,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    num_return_sequences=G,
)
prompt_len = inputs["input_ids"].shape[1]
for i in range(G):
    print(f"--- sample {i} ---")
    print(tok.decode(out[i, prompt_len:]))


--- sample 0 ---
Natalia started selling clips in April, and half of them ended up in May. 48 / 2 = 24, but she sold 16 of them in May (33% of 24). Half of 16 clips is 8, but 33% of 16 is 8 * 16/10 = 12/10 = 3. The total number of clips sold (in either April or May) is 16 + 3 = 19. Natalia sold 19 clips in April and 19 + 3 = 22 in May, accounting for 1 + 2 = 3.<|im_end|>
--- sample 1 ---
On April, Natalia sold 48/12 = 3.4 clips to her friends.
On May, when she sold half of the clips, she still had 48/2 = 24 clips left.
#### 24
#### 24
#### 24
#### 24
#### 24<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_e

In [18]:
completions = out[:, prompt_len:]
is_eos = completions == tok.eos_token_id
print(is_eos.int())
print(is_eos.int().argmax(dim=1))


tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
      

In [22]:
from grpo.sampling import sample_group

question = train[0]["question"]
completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

print(completions.shape, mask.sum(dim=1), truncated)
for i in range(2):
    text = tok.decode(completions[i][mask[i]], skip_special_tokens=True)
    print(f"--- sample {i} ---")
    print(text)


torch.Size([4, 256]) tensor([256, 104,  28, 150], device='mps:0') tensor([ True, False, False, False], device='mps:0')
--- sample 0 ---
On April, Natalia sold 17 x 48 = 784 clips, and on May, she sold a similar number of clips, but in a different order, so 48 + (6 * 1 + 6 * 2 + ... + 6 * n + 3r) = $784.

Then, 
- 6 * 1 = $6, and 
- 6 * 1 + 6 * 2 = $3, then to
- 6 * 1 + 3 + 3 * 2 = $24, then to
- 6 * 1 + 2 = $2, then to
- 6 * 2 + 2 * 1 = $16, then to
- 6 * 2 + 3 = $28, then to
- 6 * 2 + 4 = $56.

However, directly, it is an exact however (since the numbers are correct physically) 6 x 48 = 256 = 84. So all order having preserved total values by matching order and equality then replacements are not needed, it highlights the
--- sample 1 ---
In April, Natalia sold 48/4 = 125 clips.
In May, she sold half of those, so she sold 125/2 = 66 clips.
There are 36 + 66 = 102 clips in total.
There are 48 friends in total, so she sold 102/48 = 15 clips in April.
#### 15
The answer is: 15


In [23]:
question

'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [30]:
from grpo.reward import reward

question = train[0]["question"]
answer = train[0]["answer"]
input_ids, completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

texts = [tok.decode(completions[i][mask[i]], skip_special_tokens=True) for i in range(4)]
rewards = torch.tensor([reward(t, answer) for t in texts])
print(rewards)


tensor([0.0000, 0.1000, 0.1000, 0.1000])


In [25]:
texts

['Natalia sold clips to 48 people in April, and she sold half of those clips in May. So, Natalia sold approximately 48 * 5/2 = 245 clips in April.\n\nThe total number of clips sold by Natalia in both April and May is 5 people / clip sold per clip = 25 clips.\n\nNatalia sold 5/2 * 25 / 5 = 15 clips in April and 25 clips in May.\n#### 15\nThe answer is: 15',
 "First, let's determine the number of clips sold in April: Four8 canceling for 48.\nIn May, Natalia sold half as many clips: 36 canceling for 36.\n#### 36\n#### 48\n#### 48\n#### 46\n#### 54\n#### 65\n#### 70\n#### 78\n#### 85\n#### 94\n#### 100",
 'Natalia sold 48 clips to her friends, and in April, she sold 4 * 48 = 192 clips. She also sold half as many clips as in May, so she sold 192 / 2 = 98 clips in May.\n\nTo find the number of clips sold altogether in both April and May, we add the number of clips sold in April and the number of clips sold in May.\n\nC = V / (Q * (1 + R) * ((1 + R)) * S)\nC = 192 / (16 * (1 + 6)) * ((1 + 6) 

In [29]:
adv = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
adv

tensor([ 0.4990,  0.4990, -1.4970,  0.4990])

In [28]:
rewards

tensor([0.1000, 0.1000, 0.0000, 0.1000])

In [32]:
from grpo.sampling import sample_group, token_logprobs

prompt_ids, completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

with torch.no_grad():
    lp = token_logprobs(model, prompt_ids, completions)

print(lp.shape, completions.shape)
print((lp * mask).sum(dim=1) / mask.sum(dim=1))   # average log-prob per real token


torch.Size([4, 130]) torch.Size([4, 130])
tensor([-0.6012, -0.5439, -0.4071, -0.3021], device='mps:0')
